# Lab 4 — LangChain (EP02)

Lab 1–3 เราเขียนทุกอย่างเอง: ต่อ `messages` เอง, ส่ง `response_format` เอง, `model_validate_json` เอง, วน retry เอง
lab นี้เราสร้าง **ตัวคัดแยกอีเมลตัวเดิม** ใหม่ด้วย **LangChain** แล้วจับคู่ทีละชิ้นกับโค้ดที่เราเขียนเอง

| เราเขียนเอง (Lab 1–2) | LangChain |
|---|---|
| `client` + `MODEL` + `EXTRA` | `ChatOpenAI(...)` |
| `FEW_SHOT` (list ของ messages) | `ChatPromptTemplate` + `FewShotChatMessagePromptTemplate` |
| `schema_of()` + `response_format` + `model_validate_json` | `llm.with_structured_output(StudentEmail)` |
| `for email in EMAILS:` | `chain.batch([...])` (รันพร้อมกัน) |
| `extract(email, tries=3)` | `.with_retry(stop_after_attempt=3)` |

เวอร์ชันที่ใช้: `langchain-openai==1.7.2`, `langchain-core==1.7.0` (ชุดเดียวกับ EP05 Lab 5)

> อ้างอิง: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.13.5, §19.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — ฉบับให้ผู้เรียนลงมือเขียน**
>
> โค้ดส่วนใหญ่เตรียมไว้ให้แล้ว เขียนเพิ่มเฉพาะ **6 จุดที่มี `✏️ TODO`** (แทนที่ `___` หรือ `...` ด้วยโค้ดของคุณ) ทุกจุดบอกเวลาในวิดีโอที่อธิบายโค้ดตรงนั้น (▶ m:ss) — ดูคลิปก่อน แล้วลองเขียนเองโดยไม่เปิดเฉลย
>
> รันทุก cell ตามลำดับจากบนลงล่าง ถ้า cell ไหนยังเติมไม่ครบจะเจอ `NameError` หรือ `TypeError` เป็นเรื่องปกติ
>
> ติดตรงไหน เปิดเฉลยใน **`ep02-lab4-langchain.ipynb`** (โค้ดชุดเดียวกับในวิดีโอ)

## 0. เตรียมเครื่องก่อนเริ่ม

เหมือน Lab 1: Python 3.10+ หรือ Colab และ `OPENROUTER_API_KEY` ใน Colab Secrets หรือไฟล์ `.env`
(วิธีตั้งค่า key และเหตุผลที่เลือก model กับ provider อยู่ใน Lab 1)

In [ ]:
%pip install -q "langchain-openai==1.7.2" "langchain-core==1.7.0" "pydantic>=2.7" python-dotenv

In [ ]:
import os, re
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, field_validator

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

EMAILS = [
    "เรียนอาจารย์ค่ะ หนูป่วยเป็นไข้หวัดใหญ่ ขอส่งการบ้านครั้งที่ 2 วิชา AI301 ช้าไป 3 วันได้ไหมคะ แนบใบรับรองแพทย์มาด้วยค่ะ",
    "อาจารย์ครับ สอบกลางภาคออกถึงบทไหนครับ",
    "ระบบ LMS ล่มตอนผมกดส่งงาน CS214 แล้วโดนหักคะแนนส่งช้า ผมว่าไม่ยุติธรรมครับ",
    "ขอเลื่อนส่งรายงานกลุ่มเป็นวันจันทร์ได้ไหมครับ สมาชิกในกลุ่มติดสอบวิชาอื่น",
    "Lab 3 ของวิชา AI301 ต้องใช้ API key ของตัวเองหรือเปล่าคะ",
    "ผมส่งการบ้านไปสองรอบแล้วแต่คะแนนยังขึ้นศูนย์ ช่วยตรวจสอบด้วยครับ",
]

class StudentEmail(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Literal["question", "extension", "complaint"] = Field(
        description="question = ถามข้อมูล · extension = ขอส่งงานช้ากว่ากำหนด · complaint = แจ้งปัญหาหรือไม่พอใจ")
    urgency: Literal["low", "medium", "high"] = Field(description="high = มีกำหนดส่งหรือคะแนนเสีย")
    course: str | None = Field(description="รหัสวิชาที่เขียนในอีเมล เช่น AI301 ถ้าไม่มีให้เป็น null")
    summary: str = Field(description="สรุปหนึ่งประโยค")

    @field_validator("course")
    @classmethod
    def must_be_code(cls, v):
        if v is not None and not re.fullmatch(r"[A-Z]{2,3}\d{3}", v):
            raise ValueError(f"ไม่ใช่รหัสวิชา: {v!r}")
        return v

## 1. `ChatOpenAI` = client + MODEL + EXTRA

In [ ]:
from langchain_openai import ChatOpenAI

EXTRA = {"reasoning": {"enabled": False}, "provider": {"order": ["parasail/bf16"], "allow_fallbacks": False}}
# ✏️ TODO 1 (▶ วิดีโอ 1:47): สร้าง ChatOpenAI ที่ชี้ไปที่ OpenRouter (base_url) พร้อม key และ EXTRA
llm = ChatOpenAI(model="google/gemma-4-26b-a4b-it", base_url=___,
                 api_key=load_key(), extra_body=EXTRA)
print(llm.invoke("ทักทายนักศึกษาสั้นๆ หนึ่งประโยค").content)

## 2. Few-shot ด้วย prompt template (Lab 1)

`ChatPromptTemplate` = ใบสั่งงานที่มีช่องว่างรอเติม (`{email}`) · ตัวอย่างอยู่ในรูปข้อมูล ไม่ใช่ messages ที่ต่อเอง

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

EXAMPLES = [{"email": "ส่งงานช้าได้ไหมคะ พอดีติดธุระที่บ้าน", "label": "ขอเลื่อนส่ง"},
            {"email": "quiz สัปดาห์หน้าออกบทไหนครับ", "label": "คำถาม"},
            {"email": "คะแนนสอบผมหายไปจากระบบ ไม่มีใครตอบเลยครับ", "label": "ร้องเรียน"}]
# ✏️ TODO 2 (▶ วิดีโอ 2:55): บอกว่าตัวอย่างหนึ่งคู่ = อีเมลจาก human ตามด้วยป้ายจาก ai
few_shot = FewShotChatMessagePromptTemplate(
    examples=EXAMPLES,
    example_prompt=ChatPromptTemplate.from_messages([___, ___]))

In [ ]:
classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "จัดประเภทอีเมลนักศึกษาเป็นหนึ่งใน: คำถาม / ขอเลื่อนส่ง / ร้องเรียน"),
    few_shot,
    ("human", "{email}")])
# ✏️ TODO 3 (▶ วิดีโอ 3:28): ต่อสายพาน: prompt → llm → ดึงข้อความ
classify = ___

In [ ]:
# ✏️ TODO 4 (▶ วิดีโอ 3:51): ส่งอีเมลทุกฉบับเข้า chain พร้อมกันในครั้งเดียว
labels = classify.___([{"email": e} for e in EMAILS])
print(labels)

## 3. Structured output ด้วย `with_structured_output` (Lab 2)

บรรทัดเดียวแทน `schema_of()` + `response_format` + `model_validate_json`

In [ ]:
email_prompt = ChatPromptTemplate.from_messages([
    ("system", "คุณคือผู้ช่วยอาจารย์ อ่านอีเมลนักศึกษาใน <email> แล้วกรอกข้อมูลตาม schema ใช้เฉพาะข้อมูลที่อยู่ในอีเมล"),
    ("human", "<email>\n{email}\n</email>")])
# ✏️ TODO 5 (▶ วิดีโอ 4:31): ให้ llm ตอบตาม schema StudentEmail แบบ json_schema และ strict
structured = llm.___(StudentEmail, method="json_schema", strict=True)
extractor = email_prompt | structured

THAI = {"question": "คำถาม", "extension": "ขอเลื่อนส่ง", "complaint": "ร้องเรียน"}
for info in extractor.batch([{"email": e} for e in EMAILS]):
    print(THAI[info.intent], info.urgency, repr(info.course), "|", info.summary)

## 4. `.with_retry()` แทน `extract(email, tries=3)` — และสิ่งที่มันไม่ได้ทำ

`with_retry` **ส่ง input เดิมซ้ำ** เมื่อเกิด exception — ไม่ได้บอก model ว่าผิดตรงไหน
ลองกับ prompt เปล่าๆ (ไม่มี system prompt และป้าย `<email>`) ซึ่งใน Lab 2 ทำให้ course ผิดบ่อย

In [ ]:
# ✏️ TODO 6 (▶ วิดีโอ 5:43): ห่อ chain ให้ถามใหม่ได้ไม่เกิน 3 ครั้งเมื่อเกิด error
plain_prompt = ChatPromptTemplate.from_messages([("human", "อ่านอีเมลนักศึกษาแล้วสรุปข้อมูล\nอีเมล: {email}")])
retrying = (plain_prompt | structured).___(stop_after_attempt=3)
for e, out in zip(EMAILS, retrying.batch([{"email": e} for e in EMAILS], return_exceptions=True)):
    print(e[:30], "→", type(out).__name__ if isinstance(out, Exception) else repr(out.course))

## สรุป: เขียนเอง vs LangChain

| งาน | เขียนเอง (Lab 1–2) | LangChain (lab นี้) |
|---|---|---|
| เรียก model | `client` + `MODEL` + `EXTRA` + `ask()` | `ChatOpenAI(...)` |
| few-shot | list ของ messages ที่ต่อเอง | `FewShotChatMessagePromptTemplate` |
| ต่อขั้นตอน | เรียกฟังก์ชันซ้อนกัน | `prompt \| llm \| parser` |
| structured output | `schema_of()` + `response_format` + `model_validate_json` | `with_structured_output(StudentEmail)` |
| หลายอีเมล | `for` loop | `.batch()` (ขนาน) |
| ถามใหม่ | `extract(tries=3)` | `.with_retry(stop_after_attempt=3)` |

**สิ่งที่ LangChain ไม่ได้ทำให้**
- ไม่ได้เขียน prompt หรือ schema ที่ดีให้ — `StudentEmail`, คำอธิบายช่อง และ validator ยังเป็นของเรา
- `with_retry` ส่ง input เดิมซ้ำ **ไม่ได้บอก model ว่าผิดตรงไหน** (library อย่าง Instructor ส่ง error กลับให้ model แก้)
- ไม่ได้ตรวจ hallucination — การตรวจ quote ใน Lab 3 ยังเป็นโค้ดของเรา

**สิ่งที่ได้เพิ่ม:** chain เดียวกันเรียกได้ทั้ง `.invoke` (ฉบับเดียว) และ `.batch` (หลายฉบับขนานกัน)
และ chain แบบนี้คือสิ่งที่เราจะเอาไปเป็น node ใน LangGraph ใน EP05

**แบบฝึกหัด:** เขียน `extract_with_feedback(email)` ที่ถ้า validate ไม่ผ่าน ให้ต่อข้อความ error ลงใน prompt รอบถัดไป
("รอบที่แล้ว course ผิด: ...") แล้วเทียบกับ `with_retry` บน `plain_prompt`

▶ **ถัดไป: สรุป EP02**